# Fixed-pose scoring and additive evaluations

Use a DockingMT environment with a compatible Vina engine and an explicit saved input record. Set `DOCKINGMT_RESULT_PATH` and its externally expected `DOCKINGMT_RESULT_SHA256` before running all four cells; see [the contract and retained example](fixed_pose_scoring.md). No checkout location, environment name or Python minor is assumed here.

The verified captured PDBQT partner conformation is evaluated twice, without selecting a historical output pose, search or optimization. Original producer and current evaluations stay separate. Preparation/chemical information remains as declared; the retained overlapping four/two-atom control is not an accuracy or affinity benchmark. Source outputs are clean; exact executed evidence is retained separately.


In [ ]:
import base64
import gzip
import hashlib
import json
import os
import platform
from pathlib import Path

import numpy as np
import pyunitwizard as puw
from smonitor.integrations import CatalogException

import dockingmt as dmt

input_path = Path(os.environ['DOCKINGMT_RESULT_PATH']).expanduser().resolve()
expected_sha256 = os.environ['DOCKINGMT_RESULT_SHA256'].lower()
if len(expected_sha256) != 64 or any(
    char not in '0123456789abcdef' for char in expected_sha256
):
    raise ValueError('Supply an externally expected file SHA-256 digest.')
input_bytes = input_path.read_bytes()
if hashlib.sha256(input_bytes).hexdigest() != expected_sha256:
    raise ValueError('Input file SHA-256 does not match the trusted expectation.')
decoded = gzip.decompress(input_bytes) if input_path.suffix == '.gz' else input_bytes
record = json.loads(decoded)
audit = dmt.audit_result(record)
if audit['status'] == 'inconsistent':
    raise ValueError('Resolve inconsistent saved-input evidence before scoring.')
saved_input = dmt.DockingResult.from_dict(record)
artifacts = record.get('provenance', {}).get('backend_artifacts', {})
captured_sizes = dmt.verify_captured_inputs(artifacts)
if any(artifacts[role].get('format') != 'pdbqt' for role in ('receptor', 'partner')):
    raise ValueError('This example requires captured prepared PDBQT inputs.')
problem = dmt.DockingProblem(
    receptor=base64.b64decode(
        artifacts['receptor']['content_base64'], validate=True
    ).decode(),
    partner=base64.b64decode(
        artifacts['partner']['content_base64'], validate=True
    ).decode(),
    search_domain=dmt.BoxRegion.from_dict(saved_input.problem_info['search_domain']),
)
print(
    {
        'python': platform.python_version(),
        'reader': dmt.__version__,
        'recorded_input_producer': record.get('provenance', {}).get(
            'dockingmt_version'
        ),
        'input_file_sha256': expected_sha256,
        'input_audit': audit['status'],
        'captured_sizes_bytes': captured_sizes,
        'geometry': 'captured submitted partner; no saved output pose selected',
    }
)

In [ ]:
first = dmt.score(
    problem,
    dmt.VinaProtocol(cpu=1, seed=17, capture_backend_inputs=True),
    score_name='vina',
)
before = first.to_dict()
second = dmt.score(
    problem,
    dmt.VinaProtocol(cpu=1, seed=17, scoring='vinardo'),
    pose=first,
    score_name='vinardo',
)
assert first.to_dict() == before
np.testing.assert_array_equal(
    puw.get_value(first.coordinates), puw.get_value(second.coordinates)
)
assert len(second.scores) == 16
assert second.rank is None
for name, value in second.scores.items():
    definition = second.score_definitions[name]
    print(f'{name:30s} {value:10.3f} {definition["unit"]} ({definition["kind"]})')

In [ ]:
for evaluation in second.metadata['scoring_history']:
    report = {
        key: evaluation[key]
        for key in (
            'operation',
            'backend_version',
            'applied_parameters',
            'unused_search_parameters',
            'geometry_check',
        )
    }
    report['input_hashes'] = {
        role: record['sha256']
        for role, record in evaluation['backend_artifacts'].items()
    }
    report['preparation_assessment'] = {
        role: record['assessment_report']['assessment']
        for role, record in evaluation['preparation'].items()
    }
    print(json.dumps(report, indent=2))

In [ ]:
annotated = second.with_scores({'manual_review': 0.75})
assert 'manual_review' not in second.scores
assert 'manual_review' not in annotated.score_definitions  # Meaning remains undeclared.
restored = dmt.DockingPose.from_dict(
    json.loads(json.dumps(annotated.to_dict(), allow_nan=False))
)
assert restored.to_dict() == annotated.to_dict()
try:
    second.with_scores({'vina': 0.0})
except CatalogException as error:
    if error.code != 'DMT-E002':
        raise
    print(type(error).__name__ + ': an existing score cannot be replaced')
else:
    raise AssertionError('Existing scores were overwritten')
print(
    'Independent roundtrip:',
    len(restored.scores),
    'scores;',
    len(restored.metadata['scoring_history']),
    'native evaluations',
)